# Cohort-Aware Markowitz Portfolio

Builds on the K-Medoids + DTW clustering from `KMedoids_DTW.ipynb`: 241 FTSE-listed
names, split at the notebook's own consensus **k = 2** into two behavioural cohorts
(Cohort A, medoid `DLN.L`, n=103; Cohort B, medoid `ESCT.L`, n=138).

That notebook's own diagnostic found an Adjusted Rand Index of **0.795** between the
two clusters and simple sign-of-trend — i.e. the clustering is largely a restatement
of trend direction over 2021-2026, not a distinct-strategy split. So here the cohort
labels are **not** used as a return signal (that would just be performance-chasing on
one window); they're used as a **diversification guardrail** — real historical return
and risk data drive the weights, and a constraint keeps both cohorts represented.

**Input needed:** the *raw* (non-normalised) adjusted-close price CSV — dates as rows,
tickers as columns (`ftse250_adj_close_only.csv`). The z-normalised file used for
clustering only preserves shape, not scale, so it can't be used to estimate real
returns/covariance.

**Pipeline:** 241 names -> 50-name universe (Sharpe-ranked within cohort,
correlation-deduplicated) -> Bayes-Stein shrunk expected returns -> Ledoit-Wolf
shrunk covariance -> constrained mean-variance optimisation (long-only, 8%
single-name cap, cohort held to 30-70% of the book) -> efficient frontier + three
portfolios (Conservative / Balanced / Growth) -> historical backtest.

**Not investment advice** — a quantitative construction exercise on ~4.6 years of
one market regime.

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

ANN = 252  # trading days/year
print("Ready.")

## Load the raw adjusted-close price file

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    FILENAME = list(uploaded.keys())[0]
else:
    FILENAME = "ftse250_adj_close_only.csv"
print("Using file:", FILENAME)

## Cluster labels

Carried over verbatim from `KMedoids_DTW.ipynb`'s cell-14 output (the consensus
k=2 result). If you re-run that notebook with different parameters and get a
different partition, paste the new ticker lists in here.

In [ ]:
cluster0_medoid = "DLN.L"
cluster0 = """AML.L, ASHM.L, BHMG.L, BKG.L, BME.L, BPT.L, BREE.L, BRSC.L, BSIF.L, BWY.L, BYG.L, BYIT.L,
CBG.L, CGT.L, CVSG.L, CWR.L, DATA.L, DLN.L, DNLM.L, DOCS.L, DOM.L, DSCV.L, ENOG.L, EZJ.L,
FGEN.L, FGT.L, FRAS.L, GAMA.L, GBG.L, GCP.L, GEN.L, GFTU.L, GNS.L, GPE.L, GRG.L, GRI.L, GROW.L,
HAS.L, HFEL.L, HFG.L, HICL.L, HIK.L, HSL.L, IHP.L, INCH.L, INPP.L, IPO.L, ITV.L, IWG.L, JDW.L,
JUP.L, KNOS.L, MGAM.L, MNDI.L, MONY.L, N91.L, NAS.L, NBPE.L, NCC.L, OCDO.L, ONT.L, PAGE.L,
PETS.L, PHP.L, PNN.L, PTEC.L, RCP.L, RHIM.L, RICA.L, RMV.L, RS1.L, RSW.L, SAFE.L, SEIT.L,
SEQI.L, SHC.L, SMWH.L, SPI.L, SRE.L, SSPG.L, SUPR.L, SVS.L, SYNC.L, TATE.L, TEP.L, THG.L,
THRL.L, TPK.L, TRIG.L, TRN.L, TRY.L, TW.L, UKW.L, UTG.L, VCT.L, VOF.L, VTY.L, WIZZ.L, WKP.L,
WOSG.L, WPP.L, XPP.L, ZIG.L""".replace("\n", " ").replace(" ", "").split(",")

cluster1_medoid = "ESCT.L"
cluster1 = """3IN.L, AAS.L, AEP.L, AGT.L, AIE.L, AJB.L, ALFA.L, AO.L, ASL.L, ATR.L, ATT.L, ATYM.L, AVON.L,
BAG.L, BBY.L, BCG.L, BGFD.L, BMY.L, BNKR.L, BOWL.L, BOY.L, BPCR.L, BRGE.L, BRWM.L, BUT.L,
CHG.L, CKN.L, CLDN.L, CMCX.L, COA.L, CORD.L, COST.L, CSN.L, CTY.L, CURY.L, CWK.L, DRX.L,
EDIN.L, ELM.L, EMG.L, ESCT.L, EWG.L, EWI.L, FAN.L, FCH.L, FCSS.L, FEML.L, FEV.L, FGP.L,
FOUR.L, FSG.L, FSV.L, GDWN.L, GFRD.L, GNC.L, GSCT.L, HANA.L, HGT.L, HILS.L, HMSO.L, HOC.L,
HRI.L, HTG.L, HTWS.L, HVPE.L, HWG.L, IAD.L, ICGT.L, IPF.L, JAM.L, JCH.L, JEDT.L, JEGI.L,
JEMI.L, JFJ.L, JGGI.L, JMAT.L, JMGI.L, JSG.L, JTC.L, KIE.L, KLR.L, LRE.L, LWDB.L, MAB.L,
MEGP.L, MGNS.L, MNKS.L, MNTN.L, MRC.L, MRCH.L, MTO.L, MTRO.L, MUT.L, MYI.L, OCI.L, OSB.L,
OXB.L, OXIG.L, PAF.L, PAG.L, PCGH.L, PEY.L, PFD.L, PHI.L, PIN.L, PINT.L, PLUS.L, PNL.L, POLN.L,
PPET.L, PPH.L, QLT.L, QQ.L, RAT.L, RNK.L, ROR.L, RTW.L, SAGA.L, SAIN.L, SCT.L, SDP.L, SNR.L,
SOI.L, SRP.L, TBCG.L, TCAP.L, TEM.L, TFIF.L, TMPL.L, TRST.L, UEM.L, USA.L, VEIL.L, VSVS.L,
WIX.L, WWH.L, XPS.L""".replace("\n", " ").replace(" ", "").split(",")

assert len(cluster0) == 103 and len(cluster1) == 138
assert len(set(cluster0) & set(cluster1)) == 0

cluster_map = {t: 0 for t in cluster0}
cluster_map.update({t: 1 for t in cluster1})
print(f"Cohort A: {len(cluster0)} names (medoid {cluster0_medoid})")
print(f"Cohort B: {len(cluster1)} names (medoid {cluster1_medoid})")

## Step 1 — load prices, fill gaps, compute returns

In [ ]:
px = pd.read_csv(FILENAME)
px["Date"] = pd.to_datetime(px["Date"])
px = px.set_index("Date").sort_index()

assert set(px.columns) == set(cluster_map.keys()), "ticker mismatch between price file and cluster labels"

n_na_before = int(px.isna().sum().sum())
px = px.ffill().bfill()          # same treatment as the clustering notebook
assert px.isna().sum().sum() == 0

tickers_all = sorted(cluster_map.keys())
px = px[tickers_all]
rets = px.pct_change().dropna(how="all").iloc[1:]

print(f"Price matrix   : {px.shape}")
print(f"Return matrix  : {rets.shape}")
print(f"Gaps filled    : {n_na_before}")
print(f"Date range     : {px.index.min().date()} to {px.index.max().date()}")

## Step 2 — reduce the universe (241 -> 50)

Rank each cohort's members by historical Sharpe ratio, then greedily add names
whose return correlation with everything already picked stays below 0.80 — this
avoids loading the optimiser with near-duplicate movers, while keeping the two
cohorts represented in rough proportion to their size.

In [ ]:
TARGET_TOTAL = 50
CORR_THRESHOLD = 0.80

mu_raw_all = rets.mean() * ANN
sigma_raw_all = rets.std() * np.sqrt(ANN)
sharpe_raw_all = mu_raw_all / sigma_raw_all
corr_all = rets.corr()

cluster_ids = sorted(set(cluster_map.values()))
sizes = pd.Series(cluster_map).value_counts()
targets = {c: max(5, round(TARGET_TOTAL * sizes[c] / sizes.sum())) for c in cluster_ids}

selected = []
for c in cluster_ids:
    members = [t for t, cc in cluster_map.items() if cc == c]
    ranked = sharpe_raw_all[members].sort_values(ascending=False).index.tolist()
    chosen = []
    for t in ranked:
        if len(chosen) >= targets[c]:
            break
        if not chosen or corr_all.loc[t, chosen].max() < CORR_THRESHOLD:
            chosen.append(t)
    selected.extend(chosen)
    print(f"Cohort {c}: picked {len(chosen)}/{targets[c]} from {len(members)} members")

tickers = sorted(selected)
print(f"\nUniverse size: {len(tickers)}")

universe = pd.DataFrame({
    "ticker": tickers,
    "cluster": [cluster_map[t] for t in tickers],
    "ann_return": [mu_raw_all[t] for t in tickers],
    "ann_vol": [sigma_raw_all[t] for t in tickers],
    "sharpe_raw": [sharpe_raw_all[t] for t in tickers],
}).sort_values(["cluster", "sharpe_raw"], ascending=[True, False]).reset_index(drop=True)
universe

## Step 3 — estimate expected returns and covariance

**Covariance:** Ledoit-Wolf shrinkage toward a structured target — stabilises the
50x50 sample covariance that ~1,150 daily observations alone would leave noisy.

**Expected returns:** the textbook Jorion (1986) Bayes-Stein shrinkage intensity
collapses to ~0 here because the 50-name universe is still collinear enough
(same-cohort names share >0.5 correlation even after de-duplication) to inflate the
Mahalanobis dispersion of the means — that's a collinearity artefact, not evidence
that 4.6 years of daily returns pin down expected returns precisely. So a disclosed,
fixed 40% shrinkage toward the cross-sectional (minimum-variance) grand mean is used
instead.

In [ ]:
R = rets[tickers]
N, T = len(tickers), len(R)

mu_hat = R.mean().values * ANN
lw = LedoitWolf().fit(R.values)
Sigma = lw.covariance_ * ANN

Sigma_inv = np.linalg.inv(Sigma)
ones = np.ones(N)
mu_grand = (ones @ Sigma_inv @ mu_hat) / (ones @ Sigma_inv @ ones)
diff = mu_hat - mu_grand
quad = diff @ Sigma_inv @ diff
phi_formula = float(np.clip((N + 2) / ((N + 2) + T * quad), 0, 1))

PHI = 0.40  # disclosed practitioner shrinkage (see markdown above)
mu = (1 - PHI) * mu_hat + PHI * mu_grand

print(f"Universe N={N}, observations T={T}")
print(f"Ledoit-Wolf shrinkage intensity : {lw.shrinkage_:.3f}")
print(f"Grand mean (annualised)         : {mu_grand:.3%}")
print(f"Formula-implied phi (unused)    : {phi_formula:.4f}  <- degenerate, see markdown")
print(f"Practitioner phi used           : {PHI:.2f}")
print(f"Raw mean range   : [{mu_hat.min():.1%}, {mu_hat.max():.1%}]")
print(f"Shrunk mean range: [{mu.min():.1%}, {mu.max():.1%}]")

## Step 4 — constrained mean-variance optimisation

Long-only, 8% cap per name, and Cohort B held to 30-70% of the book (the guardrail
that keeps this from just being a bet on whichever cohort trended up over
2021-2026). Solved with SLSQP for the minimum-variance portfolio, the max-Sharpe
portfolio, an unconstrained max-Sharpe for comparison, and a higher-return point on
the frontier.

In [ ]:
RF = 0.04          # disclosed assumption: ~4% annual risk-free rate — change as needed
STOCK_CAP = 0.08
CLUSTER1_MIN, CLUSTER1_MAX = 0.30, 0.70

cluster_arr = universe.set_index("ticker").loc[tickers, "cluster"].values
is1 = (cluster_arr == 1).astype(float)

bounds = [(0.0, STOCK_CAP)] * N
w0 = np.ones(N) / N

def port_ret(w): return w @ mu
def port_var(w): return w @ Sigma @ w
def port_vol(w): return np.sqrt(port_var(w))
def neg_sharpe(w): return -(port_ret(w) - RF) / port_vol(w)

base_cons = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]
cluster_cons = base_cons + [
    {"type": "ineq", "fun": lambda w: (w @ is1) - CLUSTER1_MIN},
    {"type": "ineq", "fun": lambda w: CLUSTER1_MAX - (w @ is1)},
]

def solve(objective, cons, x0=w0):
    res = minimize(objective, x0, method="SLSQP", bounds=bounds, constraints=cons,
                    options={"maxiter": 1000, "ftol": 1e-12})
    if not res.success:
        raise RuntimeError(res.message)
    w = np.clip(res.x, 0, None)
    return w / w.sum()

w_gmv = solve(port_var, cluster_cons)
w_msr = solve(neg_sharpe, cluster_cons, x0=w_gmv)
w_msr_unconstrained = solve(neg_sharpe, base_cons, x0=w0)

r_lo = port_ret(w_gmv)
r_hi = float(np.max(mu[cluster_arr == 1]))
targets_r = np.linspace(r_lo, r_hi * 0.9, 40)
frontier_rows, w_prev = [], w_gmv
for rt in targets_r:
    cons = cluster_cons + [{"type": "eq", "fun": lambda w, rt=rt: port_ret(w) - rt}]
    try:
        w = solve(port_var, cons, x0=w_prev)
        frontier_rows.append({"return": port_ret(w), "vol": port_vol(w)})
        w_prev = w
    except Exception:
        continue
frontier = pd.DataFrame(frontier_rows)

vol_gmv = port_vol(w_gmv)
growth_row = frontier.iloc[(frontier["vol"] - vol_gmv * 1.9).abs().argsort()[:1]].iloc[0]
cons_growth = cluster_cons + [{"type": "eq", "fun": lambda w, rt=float(growth_row["return"]): port_ret(w) - rt}]
w_growth = solve(port_var, cons_growth, x0=w_msr)

def summarize(w, name):
    return {"name": name, "return": port_ret(w), "vol": port_vol(w),
            "sharpe": (port_ret(w) - RF) / port_vol(w), "cluster1_weight": w @ is1,
            "n_holdings": int((w > 0.0005).sum()), "eff_n": 1 / (w ** 2).sum()}

portfolios = {
    "conservative": (w_gmv, summarize(w_gmv, "Conservative (Min-Variance)")),
    "balanced": (w_msr, summarize(w_msr, "Balanced (Max Sharpe, diversified)")),
    "growth": (w_growth, summarize(w_growth, "Growth (higher target return)")),
    "unconstrained_msr": (w_msr_unconstrained, summarize(w_msr_unconstrained, "Max Sharpe, no cohort guardrail")),
}
for key, (w, s) in portfolios.items():
    print(f"{s['name']:38s} return={s['return']:.2%}  vol={s['vol']:.2%}  "
          f"sharpe={s['sharpe']:.2f}  cohortB={s['cluster1_weight']:.0%}  "
          f"holdings={s['n_holdings']}  eff_n={s['eff_n']:.1f}")

## Efficient frontier

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frontier["vol"], frontier["return"], color="0.4", lw=2, label="Efficient frontier (constrained)")

marker_style = {"conservative": ("C0", "o"), "balanced": ("C1", "o"), "growth": ("C2", "o")}
for key, (color, marker) in marker_style.items():
    s = portfolios[key][1]
    ax.scatter(s["vol"], s["return"], c=color, marker=marker, s=140, edgecolor="k", zorder=5, label=s["name"])

s_unc = portfolios["unconstrained_msr"][1]
ax.scatter(s_unc["vol"], s_unc["return"], facecolors="none", edgecolors="0.3", s=140, lw=2, zorder=5,
           label="Unconstrained max-Sharpe (comparison)")

ax.set_xlabel("Annualised volatility"); ax.set_ylabel("Annualised return")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.set_title("Efficient frontier — cohort-constrained mean-variance")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Holdings

In [ ]:
holdings = {}
for key in ["conservative", "balanced", "growth"]:
    w = portfolios[key][0]
    df = pd.DataFrame({"ticker": tickers, "cluster": cluster_arr, "weight": w})
    df = df[df.weight > 0.0005].sort_values("weight", ascending=False).reset_index(drop=True)
    df["weight_pct"] = (df["weight"] * 100).round(2)
    holdings[key] = df
    print(f"--- {portfolios[key][1]['name']} ({len(df)} holdings) ---")
    print(df[["ticker", "cluster", "weight_pct"]].to_string(index=False))
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharex=False)
for ax, key in zip(axes, ["conservative", "balanced", "growth"]):
    df = holdings[key].sort_values("weight")
    colors = ["C0" if c == 0 else "C1" for c in df["cluster"]]
    ax.barh(df["ticker"], df["weight_pct"], color=colors)
    ax.set_title(portfolios[key][1]["name"], fontsize=10)
    ax.set_xlabel("Weight %")
    ax.tick_params(axis="y", labelsize=7)
handles = [plt.Rectangle((0,0),1,1,color="C0"), plt.Rectangle((0,0),1,1,color="C1")]
fig.legend(handles, ["Cohort A", "Cohort B"], loc="upper center", ncol=2, fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

## Step 5 — backtest (in-sample — read as a consistency check, not a forecast)

Fixed weights applied to the actual daily returns over the full history (equivalent
to rebalancing back to target weights every day), against an equal-weight benchmark
across all 241 original names. This uses the same data the weights were optimised
from, so it is expected to look good — it is not evidence of future performance.

In [ ]:
def backtest(w, R_):
    daily = R_ @ w
    curve = np.cumprod(1 + daily)
    peak = np.maximum.accumulate(curve)
    dd = curve / peak - 1
    ann_ret = curve[-1] ** (ANN / len(daily)) - 1
    ann_vol = daily.std() * np.sqrt(ANN)
    sharpe = (daily.mean() * ANN - RF) / ann_vol
    return curve, dd, {"cagr": ann_ret, "ann_vol": ann_vol, "sharpe": sharpe,
                        "max_drawdown": dd.min(), "final_value": curve[-1]}

R_uni = rets[tickers].values
curves, stats = {}, {}
for key in ["conservative", "balanced", "growth", "unconstrained_msr"]:
    w = portfolios[key][0]
    curve, dd, st = backtest(w, R_uni)
    curves[key] = curve
    stats[key] = st

R_full = rets.values
daily_eq = R_full.mean(axis=1)
curve_eq = np.cumprod(1 + daily_eq)
peak_eq = np.maximum.accumulate(curve_eq)
dd_eq = curve_eq / peak_eq - 1
stats["equal_weight_241"] = {
    "cagr": curve_eq[-1] ** (ANN / len(daily_eq)) - 1,
    "ann_vol": daily_eq.std() * np.sqrt(ANN),
    "sharpe": (daily_eq.mean() * ANN - RF) / (daily_eq.std() * np.sqrt(ANN)),
    "max_drawdown": dd_eq.min(), "final_value": curve_eq[-1],
}
curves["equal_weight_241"] = curve_eq

for key, st in stats.items():
    print(f"{key:20s} CAGR={st['cagr']:.2%}  vol={st['ann_vol']:.2%}  Sharpe={st['sharpe']:.2f}  "
          f"maxDD={st['max_drawdown']:.2%}  final=£{st['final_value']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
dates = rets.index
line_style = {"conservative": "C0", "balanced": "C1", "growth": "C2"}
for key, color in line_style.items():
    ax.plot(dates, curves[key], color=color, lw=1.6, label=portfolios[key][1]["name"])
ax.plot(dates, curves["equal_weight_241"], color="0.4", lw=1.4, ls="--", label="Equal-weight benchmark (241 names)")
ax.set_ylabel("Growth of £1"); ax.set_title("Backtest — in-sample, not a forecast")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

## Summary & disclaimers

- **Risk-free rate**: 4% annual, disclosed assumption — change `RF` above and re-run.
- **No transaction costs, taxes or rebalancing frictions** are modelled.
- **The backtest is in-sample.** It reuses the same history the weights were
  estimated from, so it is not a validation of future performance — treat it as a
  consistency check that the optimiser did what it was asked, not as a forecast.
- **Cohort membership reflects trend over 2021-2026 specifically.** It is not
  guaranteed to persist or to reverse; that is exactly why cohort weight is
  constrained (30-70%) rather than left to the optimiser.
- This is a quantitative construction exercise, **not investment advice**.